# 03 - Feature Engineering
**Vehicle Predictive Maintenance Project**

---

## Purpose of This Notebook

Feature engineering is the process of transforming our cleaned raw data into meaningful inputs that a machine learning model can learn from. Raw data alone is rarely enough — models need structured, numeric representations of the patterns we want them to detect.

We are building features across five areas:
1. **Repair History Features** — how often has a vehicle been repaired, how recently, how costly
2. **Repair Category Features** — breaking down repair patterns by category (brakes, tyres, engine etc.)
3. **Mileage Features** — how hard is this vehicle being worked
4. **Vehicle Features** — age, type, make
5. **Driver Behaviour Features** — score and sub-scores already joined from cleaning
6. **MOT Features** — proximity to next MOT

The output of this notebook is a single model-ready feature table saved to `data/processed/features.csv`.

## 1. Setup & Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

PROCESSED = '../data/processed/'

master      = pd.read_csv(PROCESSED + 'master_vehicles.csv', parse_dates=['Registered Date', 'Next MOT Date'])
maintenance = pd.read_csv(PROCESSED + 'maintenance_clean.csv', parse_dates=['Invoice Date'])
mileage     = pd.read_csv(PROCESSED + 'mileage_clean.csv', parse_dates=['Period'])

print('Cleaned data loaded.')
print(f'  Master vehicles:      {master.shape[0]:,} rows')
print(f'  Maintenance records:  {maintenance.shape[0]:,} rows')
print(f'  Mileage records:      {mileage.shape[0]:,} rows')

# Reference date — we engineer features relative to this point in time
# Using the last full month of maintenance data to avoid the 2026 submission lag
REFERENCE_DATE = pd.Timestamp('2025-12-31')
print(f'\nReference date for feature engineering: {REFERENCE_DATE.date()}')

## 2. Repair History Features

### Why
A vehicle's repair history is the strongest signal we have for predicting future repairs. Vehicles that have needed frequent repairs in the past are more likely to need them again. We want to capture:

- **Total repair count** — overall volume of repairs across the vehicle's life
- **Recent repair count** — repairs in the last 12 months, which is more predictive than lifetime totals for active vehicles
- **Days since last repair** — a vehicle that was just repaired is less immediately at risk than one that hasn't been seen in two years
- **Total spend** — a proxy for overall vehicle health; high spend often signals a vehicle with recurring issues
- **Average cost per repair** — distinguishes between vehicles with many cheap repairs vs fewer expensive ones
- **Distinct categories repaired** — a vehicle that has had issues across many different systems is a broader risk

In [ ]:
# Overall repair features — lifetime
repair_lifetime = maintenance.groupby('Registration').agg(
    Total_Repairs          = ('Job Number', 'count'),
    Total_Spend            = ('Net Amount', 'sum'),
    Avg_Cost_Per_Repair    = ('Net Amount', 'mean'),
    Distinct_Categories    = ('Top Category', 'nunique'),
    Last_Repair_Date       = ('Invoice Date', 'max'),
    First_Repair_Date      = ('Invoice Date', 'min')
).reset_index()

repair_lifetime['Days_Since_Last_Repair'] = (
    REFERENCE_DATE - repair_lifetime['Last_Repair_Date']
).dt.days

print('Lifetime repair features built.')
print(repair_lifetime.describe())

In [ ]:
# Recent repair features — last 12 months only
# More predictive for active vehicles as it reflects current condition
recent_cutoff = REFERENCE_DATE - pd.DateOffset(months=12)
recent_repairs = maintenance[maintenance['Invoice Date'] >= recent_cutoff]

repair_recent = recent_repairs.groupby('Registration').agg(
    Repairs_Last_12M       = ('Job Number', 'count'),
    Spend_Last_12M         = ('Net Amount', 'sum')
).reset_index()

print(f'Vehicles with repairs in last 12 months: {len(repair_recent):,}')
print(repair_recent[['Repairs_Last_12M', 'Spend_Last_12M']].describe())

## 3. Repair Category Features

### Why
Different repair categories have very different patterns. A vehicle that has had repeated brake issues is at different risk than one with repeated electrical faults. By breaking repair history down by category we give the model much more granular signal.

We focus on the highest volume categories identified in EDA:
- Brakes
- Wheels & Tyres
- Engine
- Electrical
- Cooling
- Breakdown / Recovery (a strong indicator of overall vehicle health)

For each category we calculate repair count and total spend per vehicle.

In [ ]:
# Define the key categories to feature-engineer
KEY_CATEGORIES = [
    'Brakes',
    'Wheels & Tyres',
    'Engine',
    'Electrical',
    'Cooling',
    'Breakdown / Recovery'
]

category_features_list = []

for category in KEY_CATEGORIES:
    cat_df = maintenance[maintenance['Top Category'] == category]
    cat_agg = cat_df.groupby('Registration').agg(
        count = ('Job Number', 'count'),
        spend = ('Net Amount', 'sum'),
        last_date = ('Invoice Date', 'max')
    ).reset_index()

    # Clean category name for column naming
    col_name = category.replace(' & ', '_').replace(' / ', '_').replace(' ', '_')

    cat_agg = cat_agg.rename(columns={
        'count':     f'Repairs_{col_name}',
        'spend':     f'Spend_{col_name}',
        'last_date': f'Last_{col_name}_Date'
    })

    # Days since last repair in this category
    cat_agg[f'Days_Since_{col_name}'] = (
        REFERENCE_DATE - cat_agg[f'Last_{col_name}_Date']
    ).dt.days

    category_features_list.append(cat_agg)

print(f'Category features built for {len(KEY_CATEGORIES)} categories.')
for df in category_features_list:
    print(f'  {df.columns[1]} — {len(df):,} vehicles with at least one repair')

## 4. Mileage Features

### Why
Mileage is one of the most important predictors of mechanical wear. A van doing 200 miles a day will wear out components much faster than one doing 50. We want to capture:

- **Average daily miles (recent)** — how hard the vehicle is currently being worked, using the last 3 months of data as the most representative recent picture
- **Average daily miles (lifetime)** — overall usage intensity across the vehicle's life

Note: 91 active vehicles have no mileage data due to no tracker being fitted. These will remain null and be handled by the model.

In [ ]:
# Recent average daily miles — last 3 months
recent_mileage_cutoff = REFERENCE_DATE - pd.DateOffset(months=3)
recent_mileage = mileage[mileage['Period'] >= recent_mileage_cutoff]

mileage_recent = recent_mileage.groupby('Registration').agg(
    Avg_Daily_Miles_Recent = ('Average Miles Per Day', 'mean')
).reset_index()

# Lifetime average daily miles
mileage_lifetime = mileage.groupby('Registration').agg(
    Avg_Daily_Miles_Lifetime = ('Average Miles Per Day', 'mean')
).reset_index()

mileage_features = mileage_lifetime.merge(mileage_recent, on='Registration', how='left')

print(f'Vehicles with mileage features: {len(mileage_features):,}')
print(mileage_features[['Avg_Daily_Miles_Lifetime', 'Avg_Daily_Miles_Recent']].describe())

## 5. Vehicle & MOT Features

### Why
Vehicle characteristics like age, size, and make all influence maintenance patterns. Older vehicles break down more. Larger vans tend to do more intensive work. Certain makes may have known reliability patterns in this fleet.

For the model we need to encode categorical variables (Make, Asset Type) as numbers since most ML algorithms cannot work with raw text. We use label encoding here — a simple integer mapping — which works well for tree-based models like XGBoost.

MOT proximity is a useful standalone signal. A vehicle approaching its MOT is more likely to have underlying issues flagged, and we know from the maintenance data that MOT is itself a repair category.

In [ ]:
from sklearn.preprocessing import LabelEncoder

vehicle_features = master[[
    'Registration', 'Asset Type', 'Vehicle Status', 'Make',
    'Vehicle Age (Years)', 'DriverScore', 'DeccelerationRate',
    'AccelerationRate', 'SpeedingRate', 'CorneringRate',
    'Days Until MOT', 'Cumulative Miles'
]].copy()

# MOT flag — due within 90 days
vehicle_features['MOT_Due_90_Days'] = (
    vehicle_features['Days Until MOT'].between(0, 90)
).astype(int)

# MOT overdue flag
vehicle_features['MOT_Overdue'] = (
    vehicle_features['Days Until MOT'] < 0
).astype(int)

# Label encode categorical columns
le = LabelEncoder()
for col in ['Asset Type', 'Make']:
    vehicle_features[f'{col}_Encoded'] = le.fit_transform(
        vehicle_features[col].fillna('Unknown')
    )
    print(f'{col} encoding mapping:')
    for i, cls in enumerate(le.classes_):
        print(f'  {i} = {cls}')
    print()

print('Vehicle and MOT features built.')

## 6. Assemble the Full Feature Table

### Why
We now join all feature groups together on Registration to create a single wide table — one row per vehicle, one column per feature. This is the format the models expect.

All joins are left joins from the vehicle_features base table, so every vehicle is retained regardless of whether it has data in each feature group. Missing values (e.g. a vehicle with no brake repairs) are filled with 0 for count/spend columns, and left as null for date-derived columns — the models can handle sparse features for categories a vehicle has never needed.

In [ ]:
# Start with vehicle features as the base
features = vehicle_features.copy()

# Join lifetime repair features
features = features.merge(
    repair_lifetime[['Registration', 'Total_Repairs', 'Total_Spend',
                     'Avg_Cost_Per_Repair', 'Distinct_Categories', 'Days_Since_Last_Repair']],
    on='Registration', how='left'
)

# Join recent repair features
features = features.merge(
    repair_recent,
    on='Registration', how='left'
)

# Join category features
for cat_df in category_features_list:
    # Only join the count, spend and days-since columns (drop the raw date column)
    date_cols = [c for c in cat_df.columns if c.endswith('_Date')]
    cat_df_slim = cat_df.drop(columns=date_cols)
    features = features.merge(cat_df_slim, on='Registration', how='left')

# Join mileage features
features = features.merge(mileage_features, on='Registration', how='left')

print(f'Feature table assembled: {features.shape[0]:,} rows x {features.shape[1]} columns')

In [ ]:
# Fill count and spend nulls with 0
# A null here means the vehicle has never had that type of repair — 0 is the correct value
count_spend_cols = (
    [c for c in features.columns if c.startswith('Repairs_')] +
    [c for c in features.columns if c.startswith('Spend_')] +
    ['Total_Repairs', 'Total_Spend', 'Avg_Cost_Per_Repair',
     'Distinct_Categories', 'Repairs_Last_12M', 'Spend_Last_12M']
)

for col in count_spend_cols:
    if col in features.columns:
        features[col] = features[col].fillna(0)

print('Zero-fill applied to count and spend columns.')
print(f'\nRemaining nulls after fill:')
nulls = features.isnull().sum()
print(nulls[nulls > 0])

In [ ]:
#Looking at active vehicles only, as these are the ones we will predict on — retired vehicles won't have recent repairs or mileage
active_features = features[features['Vehicle Status'] == 'Current - On Road']
print(f'Active vehicles: {len(active_features)}')
print('\nNulls for active vehicles:')
nulls = active_features.isnull().sum()
print(nulls[nulls > 0])

## 7. Feature Distributions & Sense Check

### Why
Before saving we do a quick visual check on the key features. This helps catch anything unexpected — extreme outliers, features that are almost entirely zero, or distributions that suggest something went wrong in the join.

In [ ]:
# Key numeric features to plot
plot_cols = [
    'Vehicle Age (Years)', 'Total_Repairs', 'Total_Spend',
    'Repairs_Last_12M', 'Days_Since_Last_Repair',
    'DriverScore', 'Avg_Daily_Miles_Recent'
]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    if col in features.columns:
        data = features[col].dropna()
        sns.histplot(data, bins=40, kde=True, ax=axes[i])
        axes[i].set_title(col)
        axes[i].set_xlabel('')

# Hide unused subplot
axes[-1].set_visible(False)

plt.suptitle('Key Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/reports/03_feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# Log transform skewed repair/spend columns (add 1 to handle zeros)
for col in ['Total_Repairs', 'Total_Spend', 'Repairs_Last_12M', 'Spend_Last_12M']:
    features[f'{col}_Log'] = np.log1p(features[col])

print('Log transformed features added.')

In [ ]:
# Correlation heatmap — helps identify highly correlated features
# Highly correlated features can cause issues in some models
numeric_cols = features.select_dtypes(include=[np.number]).columns.tolist()
corr_cols = [c for c in numeric_cols if c not in ['Asset Type_Encoded', 'Make_Encoded',
                                                    'MOT_Due_90_Days', 'MOT_Overdue']]

corr_matrix = features[corr_cols].corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0,
            linewidths=0.5, fmt='.1f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../outputs/reports/03_correlation_heatmap.png', dpi=150)
plt.show()

# Flag any very high correlations (>0.9) which might indicate redundant features
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.9:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j],
                              round(corr_matrix.iloc[i, j], 2)))

if high_corr:
    print('Highly correlated feature pairs (>0.9):')
    for a, b, v in high_corr:
        print(f'  {a}  <-->  {b}  ({v})')
else:
    print('No feature pairs with correlation > 0.9 found.')

In [ ]:
# Drop original skewed columns — log versions are cleaner for modelling
cols_to_drop = ['Total_Repairs', 'Total_Spend', 'Repairs_Last_12M', 'Spend_Last_12M']
features = features.drop(columns=cols_to_drop)
print('Original skewed columns dropped, log versions retained.')

In [ ]:
# Quick summary stats on the full feature table
print('Feature table summary:')
display(features.describe().T)

## 8. Save Feature Table

### Why
We save two versions:
- **features.csv** — all vehicles including disposed, used for model training
- **features_active.csv** — active vehicles only (Current - On Road), used for the prediction output

Keeping both means we can train on the full historical dataset but only generate predictions for vehicles currently in the fleet.

In [ ]:
# Save all vehicles
features.to_csv(PROCESSED + 'features.csv', index=False)

# Save active only
features_active = features[features['Vehicle Status'] == 'Current - On Road'].copy()
features_active.to_csv(PROCESSED + 'features_active.csv', index=False)

print('Feature tables saved:')
print(f'  features.csv         — {features.shape[0]:,} vehicles (all incl. disposed)')
print(f'  features_active.csv  — {features_active.shape[0]:,} vehicles (active only)')
print(f'  Total features:      {features.shape[1]} columns')
print()
print('Columns in feature table:')
for col in features.columns:
    print(f'  {col}')

## 9. Feature Engineering Summary

In [ ]:
print("""
========================================================
 FEATURE ENGINEERING SUMMARY
========================================================

REPAIR HISTORY FEATURES (lifetime):
  - Total_Repairs
  - Total_Spend
  - Avg_Cost_Per_Repair
  - Distinct_Categories
  - Days_Since_Last_Repair

REPAIR HISTORY FEATURES (last 12 months):
  - Repairs_Last_12M
  - Spend_Last_12M

CATEGORY FEATURES (per key category — count, spend, days since):
  - Brakes
  - Wheels & Tyres
  - Engine
  - Electrical
  - Cooling
  - Breakdown / Recovery

MILEAGE FEATURES:
  - Avg_Daily_Miles_Lifetime
  - Avg_Daily_Miles_Recent (last 3 months)
  - Cumulative Miles

VEHICLE FEATURES:
  - Vehicle Age (Years)
  - Asset Type (encoded)
  - Make (encoded)

DRIVER BEHAVIOUR FEATURES:
  - DriverScore
  - DeccelerationRate
  - AccelerationRate
  - SpeedingRate
  - CorneringRate

MOT FEATURES:
  - Days Until MOT
  - MOT_Due_90_Days (binary flag)
  - MOT_Overdue (binary flag)

OUTPUTS:
  - features.csv         (all vehicles — for training)
  - features_active.csv  (active only — for predictions)
========================================================
""")